# 🌲 Module 15: Spatial-Temporal Time-Series Analysis & Trend Forecasting

Welcome to **Module 15** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Multi-Temporal Trajectory Dynamics**: Model vegetation index curves over continuous multi-year time-series.
2. **Harmonic Seasonal Decomposition**: Separate long-term inter-annual trends from natural annual phenological cycles.
3. **Structural Breakpoint Detection**: Implement LandTrendr / BFAST break algorithms to pinpoint exact dates of canopy collapse.
4. **Trend Forecasting**: Project canopy trajectories forward with 95% confidence intervals.
5. **Deforestation Vulnerability Index (DVI)**: Compute spatial frontier risk maps to identify forest zones facing imminent clearing.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_15_timeseries_analysis.trajectory_modeler import VegetationTrajectoryModeler
from modules.module_15_timeseries_analysis.risk_forecaster import DeforestationRiskForecaster

%matplotlib inline
modeler = VegetationTrajectoryModeler(period=12, harmonic_order=2)
forecaster = DeforestationRiskForecaster(forecast_horizon=12)
print("✅ Module 15 Time-Series Environment Ready!")

## 1. Simulate Multi-Temporal Trajectories & Detect Breakpoints

We analyze 4 ecological trajectories: Stable Forest, Abrupt Clearcut, Fire with Regrowth, and Progressive Degradation.

In [ ]:
from run_module_15 import generate_synthetic_36mo_trajectories
trajectories = generate_synthetic_36mo_trajectories()
months = trajectories["months"]

for name in ["Stable Intact Forest", "Abrupt Clearcut", "Wildfire with Regrowth", "Progressive Degradation"]:
    vals = trajectories[name]
    breaks = modeler.detect_breakpoints(vals, time_steps=months, min_distance=4, drop_threshold=0.20)
    if len(breaks) > 0:
        b = breaks[0]
        print(f"{name:<24}: Break at Month {int(b['break_time'])} (Drop: -{b['drop_magnitude']:.2f}) -> {b['regime']}")
    else:
        print(f"{name:<24}: No abrupt break detected (Stable / Continuous trend)")

## 2. Harmonic Seasonal Decomposition & 12-Month Predictive Forecast

Decomposing progressive degradation into Trend + Seasonality, then projecting 12 months ahead.

In [ ]:
vals_deg = trajectories["Progressive Degradation"]
trend, season, res = modeler.fit_harmonic_seasonality(months, vals_deg)
forecast = forecaster.forecast_trajectory(months, vals_deg, horizon=12)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
ax1.plot(months, vals_deg, 'k-', label="Observed NDVI")
ax1.plot(months, trend, 'r--', label=f"Trend (Slope: {forecast['projected_slope']:.4f}/mo)")
ax1.plot(months, trend + season, 'b-', label="Fitted (Trend + Season)")
ax1.set_title("Harmonic Seasonal Decomposition")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.plot(forecast["historical_t"], forecast["historical_values"], 'k-o', label="Observed", markersize=3)
ax2.plot(forecast["future_t"], forecast["forecast"], 'r--o', label="12-Mo Forecast", markersize=3)
ax2.fill_between(forecast["future_t"], forecast["lower_95"], forecast["upper_95"], color="red", alpha=0.2, label="95% CI")
ax2.set_title("Canopy Trajectory Projection (Next 12 Months)")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()